# 04 — Epistasis Post-Processing

This notebook post-processes PLINK epistasis results from the `all_x_qtl` workflow.

Main goals:
- load raw SNP-level epistasis results from PLINK
- define genome-wide significance
- identify which SNP in each interaction is the QTL-side anchor
- isolate QTL × modifier interactions
- summarize modifier-side interaction hotspots
- collapse redundant modifier SNPs into broader loci
- map QTL intervals and modifier loci back to genomic annotations
- optionally perform deeper drug-specific inspection of the strongest modules

Important:
- Early sections are generic and should run for either FLU or PULV.
- Later sections are optional deep-dives and may contain drug-specific biological interpretation.

## 1. Parameters and file paths

This block defines which drug and epistasis mode are being analyzed.

Under the hood:
- `DRUG` selects the phenotype-specific output files.
- `RUN_LABEL` selects the epistasis design, such as `all_x_qtl`.
- All downstream file paths are built from these two settings.

Edit only this block when switching analyses.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# =========================
# EDIT ONLY THIS BLOCK
# =========================
DRUG = "FLU"                 # "FLU" or "PULV"
RUN_LABEL = "all_x_qtl"       # "all_x_qtl" or "ldpruned_x_qtl"

# =========================
# Paths (do not edit below)
# =========================
BASE_DIR = Path("/blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_epistasis")
DATA_DIR = BASE_DIR / "data"

drug_code = {"FLU": "flu", "PULV": "pulv"}[DRUG]

EPI_QT = DATA_DIR / f"epi_{drug_code}_{RUN_LABEL}.epi.qt"
SUMMARY_FILE = DATA_DIR / f"epi_{drug_code}_{RUN_LABEL}.epi.qt.summary"

SNP_INFO_TSV = DATA_DIR / "snp_info_complete.tsv"

QTL_RESULTS_TSV = {
    "FLU":  DATA_DIR / "Fluconazole_QTL_genes_20260312.csv",
    "PULV": DATA_DIR / "Pulvinatal_QTL_genes_20260330.csv",
}[DRUG]

print("DRUG:", DRUG)
print("RUN_LABEL:", RUN_LABEL)
print("Epistasis file:", EPI_QT)

DRUG: FLU
RUN_LABEL: all_x_qtl
Epistasis file: /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_epistasis/data/epi_flu_all_x_qtl.epi.qt


## 2. Load PLINK epistasis output

This file contains the raw SNP-level interaction results produced by PLINK.

For `all_x_qtl`, each row corresponds to a tested interaction between:
- one SNP from the QTL-side set
- one SNP from the genome-wide background

At this stage, the table is still purely SNP-level and contains substantial LD redundancy.

In [2]:
epi = pd.read_csv(EPI_QT, sep=r"\s+")

print("Total interaction rows:", epi.shape[0])
print("Minimum P-value observed:", epi["P"].min())

Total interaction rows: 4665905
Minimum P-value observed: 4.622e-133


## 3. Compute the genome-wide significance threshold

PLINK reports the total number of valid tests in the `.summary` file.

Under the hood:
- we sum the total number of tests across summary rows
- apply Bonferroni correction as `0.05 / total_tests`
- this defines the genome-wide significance cutoff for downstream filtering

In [3]:
summary = pd.read_csv(SUMMARY_FILE, sep=r"\s+", engine="python")

required = {"N_TOT"}
missing = required - set(summary.columns)
if missing:
    raise ValueError(f"SUMMARY_FILE missing columns: {missing}. Found: {list(summary.columns)}")

n_tests = int(summary["N_TOT"].sum())
bonf_threshold = 0.05 / n_tests

print("Summary rows (QTL set SNPs):", summary.shape[0])
print("Total tests (sum N_TOT):", n_tests)
print("Bonferroni threshold:", bonf_threshold)

Summary rows (QTL set SNPs): 2825
Total tests (sum N_TOT): 117179210
Bonferroni threshold: 4.266968517708901e-10


## Filter to Genome-Wide Significant Interactions

We now apply the Bonferroni threshold to the full PLINK epistasis output (`.epi.qt`) to keep only genome-wide significant SNP–SNP interaction tests. This reduces the dataset to a manageable size for downstream role-classification and interpretation.

In [4]:
# Code Cell 4 — Bonferroni filter
epi_sig = epi[epi["P"] < bonf_threshold].copy()

print("Genome-wide significant interactions:", epi_sig.shape[0])
print("Min P among significant:", epi_sig["P"].min())
print("Max STAT among significant:", epi_sig["STAT"].max())

Genome-wide significant interactions: 2020670
Min P among significant: 4.622e-133
Max STAT among significant: 602.769


## 4. Reconstruct the QTL SNP set and load SNP metadata

This section rebuilds the set of SNPs that fall inside the QTL intervals.

Why this matters:
- later we need to determine, for each epistatic SNP pair, which SNP belongs to the QTL-side anchor set
- we also load `snp_info`, which provides chromosome and coordinate information for SNPs

In [5]:
# Code Cell 5 — Load exact QTL SNP set used in PLINK run

SET_FILE = DATA_DIR / f"{drug_code}_{RUN_LABEL}.set"

qtl_snps = set()
current_set = None

with open(SET_FILE) as f:
    for line in f:
        tok = line.strip()
        if not tok:
            continue
        if tok == "END":
            current_set = None
            continue
        if current_set is None:
            current_set = tok
            continue
        if current_set == "QTL":
            qtl_snps.add(tok)

qtl_snps = {s for s in qtl_snps if s and s.lower() != "nan"}

print("QTL SNPs loaded from set file:", len(qtl_snps))
print("Set file:", SET_FILE)

QTL SNPs loaded from set file: 2825
Set file: /blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_epistasis/data/flu_all_x_qtl.set


## Classify Which SNP in Each Pair Belongs to the QTL Set

PLINK's `--epistasis set-by-all` does NOT guarantee that the QTL SNP
appears in SNP2. SNP pairs are ordered by genomic position.

Therefore, we explicitly determine:

- SNP1_is_qtl
- SNP2_is_qtl

This allows us to:
1. Detect QTL × QTL interactions
2. Detect genome × genome interactions
3. Properly assign QTL vs modifier roles

In [6]:
# Code Cell 6 — Classify interaction roles

epi_sig["SNP1_is_qtl"] = epi_sig["SNP1"].isin(qtl_snps)
epi_sig["SNP2_is_qtl"] = epi_sig["SNP2"].isin(qtl_snps)

# Diagnostic proportions
xor_prop = (epi_sig["SNP1_is_qtl"] ^ epi_sig["SNP2_is_qtl"]).mean()
both_qtl = (epi_sig["SNP1_is_qtl"] & epi_sig["SNP2_is_qtl"]).sum()
neither_qtl = (~epi_sig["SNP1_is_qtl"] & ~epi_sig["SNP2_is_qtl"]).sum()

print("Proportion with exactly one QTL SNP:", xor_prop)
print("QTL × QTL interactions:", both_qtl)
print("Genome × Genome interactions:", neither_qtl)

Proportion with exactly one QTL SNP: 0.6416703370664185
QTL × QTL interactions: 724066
Genome × Genome interactions: 0


## Remove QTL × QTL Interactions

We retain only interactions where exactly one SNP belongs to the QTL set.

This removes:
- QTL × QTL interactions
- Ensures all remaining rows represent QTL × genome modifier pairs

In [7]:
# Code Cell 7 — Keep only QTL × genome interactions

epi_clean = epi_sig[
    epi_sig["SNP1_is_qtl"] ^ epi_sig["SNP2_is_qtl"]
].copy()

print("QTL × Genome interactions retained:", epi_clean.shape[0])
print("Removed QTL × QTL interactions:", epi_sig.shape[0] - epi_clean.shape[0])

QTL × Genome interactions retained: 1296604
Removed QTL × QTL interactions: 724066


## Assign Explicit Roles: QTL_SNP and MOD_SNP

Because PLINK does not guarantee QTL SNP ordering,
we now explicitly assign:

- QTL_SNP → SNP inside QTL interval
- MOD_SNP → genome-wide modifier SNP

This is done using vectorized logic (fast and reproducible).

In [8]:
# Code Cell 8 — Assign roles

epi_clean["QTL_SNP"] = np.where(
    epi_clean["SNP1_is_qtl"],
    epi_clean["SNP1"],
    epi_clean["SNP2"]
)

epi_clean["MOD_SNP"] = np.where(
    epi_clean["SNP1_is_qtl"],
    epi_clean["SNP2"],
    epi_clean["SNP1"]
)

print("Unique QTL SNPs involved:", epi_clean["QTL_SNP"].nunique())
print("Unique modifier SNPs involved:", epi_clean["MOD_SNP"].nunique())

Unique QTL SNPs involved: 1518
Unique modifier SNPs involved: 4045


In [9]:
print("Total rows in epi:", epi.shape[0])
print("Genome-wide significant rows:", epi_sig.shape[0])
print("Rows with exactly one QTL SNP:", epi_clean.shape[0])
print("Unique QTL SNPs:", epi_clean["QTL_SNP"].nunique())
print("Unique modifier SNPs:", epi_clean["MOD_SNP"].nunique())

print("\nTop 10 strongest interactions:")
display(
    epi_clean.nsmallest(10, "P")[
        ["QTL_SNP", "MOD_SNP", "CHR1", "CHR2", "BETA_INT", "STAT", "P"]
    ]
)

Total rows in epi: 4665905
Genome-wide significant rows: 2020670
Rows with exactly one QTL SNP: 1296604
Unique QTL SNPs: 1518
Unique modifier SNPs: 4045

Top 10 strongest interactions:


,QTL_SNP,MOD_SNP,CHR1,CHR2,BETA_INT,STAT,P
604869,snp14442,snp34733,7,14,0.032835,587.871,8.028000e-130
599320,snp14441,snp34733,7,14,0.032813,587.136,1.160000e-129
593774,snp14440,snp34733,7,14,0.032797,586.623,1.500000e-129
543690,snp14431,snp34733,7,14,0.032697,581.918,1.582000e-128
588216,snp14439,snp34733,7,14,0.032679,581.904,1.593000e-128
582661,snp14438,snp34733,7,14,0.032676,581.794,1.684000e-128
610382,snp14443,snp34733,7,14,0.032680,581.719,1.748000e-128
549293,snp14432,snp34733,7,14,0.032684,581.496,1.955000e-128
577095,snp14437,snp34733,7,14,0.032655,581.113,2.368000e-128
565969,snp14435,snp34733,7,14,0.032652,581.017,2.485000e-128


## Inspect Interaction Effect Sizes

We examine the distribution of |BETA_INT| to understand
the magnitude of epistatic effects.

This helps determine whether effects are:
- Small but widespread
- Or driven by a few large interaction terms

In [10]:
# Code Cell 9 — Effect size distribution

epi_clean["BETA_INT"].abs().describe()

count    1.296604e+06
mean     1.007971e-02
std      3.196053e-03
min      6.908130e-03
25%      8.123395e-03
50%      9.191325e-03
75%      1.092090e-02
max      3.283550e-02
Name: BETA_INT, dtype: float64

## Identify Modifier SNP Hubs

We count how many significant interactions each modifier SNP participates in.

This reveals whether:
- Epistasis is diffuse (many SNPs, few interactions each)
- Or structured (hub modifiers interacting with many QTL SNPs)

In [11]:
# Code Cell 10 — Modifier summary

mod_summary = (
    epi_clean
    .groupby("MOD_SNP")
    .agg(
        n_interactions=("P", "count"),
        min_p=("P", "min"),
        max_beta=("BETA_INT", lambda x: x.abs().max())
    )
    .sort_values("n_interactions", ascending=False)
)

mod_summary.head(15)

,n_interactions,min_p,max_beta
MOD_SNP,,,
snp34744,1497,7.029000e-127,0.032464
snp34743,1491,1.604000e-126,0.032408
snp34742,1491,1.344000e-127,0.032559
snp34741,1491,1.344000e-127,0.032559
snp34740,1481,2.310000e-126,0.032428
snp34735,1470,3.837000e-128,0.032655
snp34734,1445,1.946000e-127,0.032521
snp34737,1444,3.542000e-127,0.032552
snp34738,1444,5.128000e-127,0.032524


In [12]:
# Code Cell 11 — Load SNP annotation

SNP_INFO_PATH = Path("/blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_epistasis/data/snp_info_complete.tsv")

snp_info = pd.read_csv(SNP_INFO_PATH, sep="\t")

print("snp_info shape:", snp_info.shape)
print(snp_info.columns.tolist())
snp_info.head()

snp_info shape: (41594, 12)
['Unnamed: 0', 'Index', 'SNP', 'Chromosome', 'Position (bp)', 'BY allele', 'RM allele', 'feature_type', 'gene_name', 'gene_chr_start', 'gene_chr_end', 'gene_standard_name']


,Unnamed: 0,Index,SNP,Chromosome,Position (bp),BY allele,RM allele,feature_type,gene_name,gene_chr_start,gene_chr_end,gene_standard_name
0,0,0,snp1,1,27210,A,G,genic,YAL063C,24000,27968,FLO9
1,1,1,snp2,1,27290,C,A,genic,YAL063C,24000,27968,FLO9
2,2,2,snp3,1,27356,T,C,genic,YAL063C,24000,27968,FLO9
3,3,3,snp4,1,27357,A,G,genic,YAL063C,24000,27968,FLO9
4,4,4,snp5,1,27370,G,A,genic,YAL063C,24000,27968,FLO9


## Map Modifier SNPs to Genomic Coordinates

To collapse LD-expanded SNP clusters into genomic loci,
we merge modifier SNPs with chromosome and position information.

In [13]:
# Code Cell 12 — Annotate modifier SNPs

mod_summary = mod_summary.reset_index()

mod_annot = mod_summary.merge(
    snp_info[["SNP", "Chromosome", "Position (bp)"]],
    left_on="MOD_SNP",
    right_on="SNP",
    how="left"
)

mod_annot.head(15)

,MOD_SNP,n_interactions,min_p,max_beta,SNP,Chromosome,Position (bp)
0,snp34744,1497,7.029000e-127,0.032464,snp34744,14,473643
1,snp34743,1491,1.604000e-126,0.032408,snp34743,14,473298
2,snp34742,1491,1.344000e-127,0.032559,snp34742,14,472579
3,snp34741,1491,1.344000e-127,0.032559,snp34741,14,472575
4,snp34740,1481,2.310000e-126,0.032428,snp34740,14,471762
5,snp34735,1470,3.837000e-128,0.032655,snp34735,14,469865
6,snp34734,1445,1.946000e-127,0.032521,snp34734,14,469219
7,snp34737,1444,3.542000e-127,0.032552,snp34737,14,470325
8,snp34738,1444,5.128000e-127,0.032524,snp34738,14,470435
9,snp34739,1443,1.444000e-127,0.032584,snp34739,14,470839


## Collapse Modifier SNPs into Genomic Regions

SNP-level counts are inflated by LD.

We cluster modifier SNPs by:
- Chromosome
- Proximity (within a defined window, e.g., 10 kb)

Each cluster represents one independent modifier locus.

In [14]:
# Code Cell 13 — Collapse modifier SNPs into loci

# Sort by chromosome and position
mod_annot_sorted = mod_annot.sort_values(
    ["Chromosome", "Position (bp)"]
).reset_index(drop=True)

# Define clustering window (10 kb)
WINDOW = 10_000

loci = []
current_chr = None
current_start = None
current_end = None

for _, row in mod_annot_sorted.iterrows():
    chr_ = row["Chromosome"]
    pos = row["Position (bp)"]

    if current_chr is None:
        current_chr = chr_
        current_start = pos
        current_end = pos
    elif chr_ == current_chr and pos - current_end <= WINDOW:
        current_end = pos
    else:
        loci.append((current_chr, current_start, current_end))
        current_chr = chr_
        current_start = pos
        current_end = pos

# Append final locus
if current_chr is not None:
    loci.append((current_chr, current_start, current_end))

print("Independent modifier loci (10kb clustering):", len(loci))
loci[:10]

Independent modifier loci (10kb clustering): 15


[(7, 329109, 452796),
 (7, 516638, 678180),
 (7, 754648, 754648),
 (7, 785199, 800053),
 (7, 832207, 903446),
 (8, 57484, 101018),
 (8, 140063, 151001),
 (12, 611852, 708594),
 (13, 873664, 917820),
 (14, 356355, 574037)]

## Summarize Modifier Loci Strength

We aggregate interaction statistics at the locus level
to determine which modifier regions are strongest.

In [15]:
# Code Cell 13 — Assign locus IDs

# Create a mapping from SNP to locus
locus_map = {}

for i, (chr_, start, end) in enumerate(loci):
    snps_in_locus = mod_annot[
        (mod_annot["Chromosome"] == chr_) &
        (mod_annot["Position (bp)"] >= start) &
        (mod_annot["Position (bp)"] <= end)
    ]["MOD_SNP"].tolist()
    
    for s in snps_in_locus:
        locus_map[s] = i

epi_clean["MOD_LOCUS"] = epi_clean["MOD_SNP"].map(locus_map)

# Aggregate
locus_summary = (
    epi_clean
    .groupby("MOD_LOCUS")
    .agg(
        n_interactions=("P", "count"),
        min_p=("P", "min"),
        max_beta=("BETA_INT", lambda x: x.abs().max())
    )
    .sort_values("n_interactions", ascending=False)
)

locus_summary.head(10)

,n_interactions,min_p,max_beta
MOD_LOCUS,,,
9,569006,8.028000e-130,0.032835
1,151931,1.179000e-106,0.029956
0,121076,7.735000e-100,0.029101
12,117797,3.215000e-22,0.013816
5,116712,9.960000e-30,0.016619
7,87531,5.508000e-19,0.022139
10,54899,2.301000e-20,0.010578
13,46194,4.676000e-14,0.010612
14,18988,2.874000e-19,0.013287


## Map Locus IDs to Chromosomal Coordinates

We now annotate each modifier locus with:
- Chromosome
- Start
- End

In [16]:
# Code Cell 14 — Annotate loci

locus_table = []

for i, (chr_, start, end) in enumerate(loci):
    locus_table.append({
        "MOD_LOCUS": i,
        "Chromosome": chr_,
        "Start": start,
        "End": end
    })

locus_table = pd.DataFrame(locus_table)

locus_table.merge(locus_summary.reset_index(), on="MOD_LOCUS").sort_values(
    "n_interactions", ascending=False
).head(10)

,MOD_LOCUS,Chromosome,Start,End,n_interactions,min_p,max_beta
9,9,14,356355,574037,569006,8.028000e-130,0.032835
1,1,7,516638,678180,151931,1.179000e-106,0.029956
0,0,7,329109,452796,121076,7.735000e-100,0.029101
12,12,15,82213,199783,117797,3.215000e-22,0.013816
5,5,8,57484,101018,116712,9.960000e-30,0.016619
7,7,12,611852,708594,87531,5.508000e-19,0.022139
10,10,14,598171,648903,54899,2.301000e-20,0.010578
13,13,15,469093,508802,46194,4.676000e-14,0.010612
14,14,15,657980,713926,18988,2.874000e-19,0.013287
8,8,13,873664,917820,8467,6.453000e-20,0.013278


## Collapse Interactions by QTL Interval (Locus-Level QTL Summary)

So far we collapsed **modifier SNPs** into modifier loci.  
Now we collapse the **QTL side** by mapping each `QTL_SNP` back to the QTL interval(s) it falls in.

This gives a QTL-interval–level view:
- Which QTL intervals are epistatically active?
- How many significant interactions does each interval have?
- Which modifier loci interact most strongly with each QTL interval?

We do this by building an interval lookup table from the QTL results file and assigning each `QTL_SNP` to an `interval_id`.

In [17]:
# Load QTL interval table
QTL_RESULTS_TSV = DATA_DIR / "Fluconazole_QTL_genes_20260312.csv"   # adjust filename if needed

qtl_df = pd.read_csv(QTL_RESULTS_TSV)
print("qtl_df shape:", qtl_df.shape)
print(qtl_df.columns.tolist())
qtl_df.head()

qtl_df shape: (45, 9)
['interval_id', 'chrom', 'start', 'end', 'genes', 'genes_std', 'pleio_score', 'pleio_percentile', 'lod_max']


,interval_id,chrom,start,end,genes,genes_std,pleio_score,pleio_percentile,lod_max
0,6,16,931074,60526,NaN,NaN,NaN,NaN,28.698400
1,268,1,46030,53935,"YAL053W, YAL051W, YAL049C, YAL048C, YAL047W-A","FLC2, OAF1, AIM2, GEM1, GEM1",0.035920,98.185543,39.443184
2,7159,4,548896,584912,"YDR046C, YDR047W, YDR049W, YDR050C, YDR051C, Y...","BAP3, HEM12, VMS1, TPI1, DET1, DBF4, CDC34, PS...",0.021940,92.029781,16.060537
3,10486,5,372639,389528,"YER105C, YER106W, YER107C, YER107W-A, YER110C,...","NUP157, MAM1, GLE2, GLE2, KAP123, SWI4, LSM4, ...",0.075441,99.783069,21.274149
4,10615,5,339948,434766,"YER091C, YER091C-A, YER093C, YER093C-A, YER094...","MET6, IES5, TSC11, AIM11, PUP3, SHC1, YNCE0018...",0.075441,99.783069,14.248954


In [18]:
# Code Cell 15— Map QTL_SNP -> interval_id, then summarize by interval

# 1) Build a SNP -> interval_id mapping by intersecting snp_info with QTL intervals
qtl_df_int = qtl_df.copy()
qtl_df_int["chrom"] = qtl_df_int["chrom"].astype(int)
qtl_df_int["start"] = qtl_df_int["start"].astype(int)
qtl_df_int["end"]   = qtl_df_int["end"].astype(int)

# Keep only the SNPs that actually appear in epi_clean to make this fast
qtl_snps_in_epi = pd.Series(epi_clean["QTL_SNP"].unique(), name="SNP")
qtl_snps_in_epi = qtl_snps_in_epi.to_frame()

qtl_snps_pos = qtl_snps_in_epi.merge(
    snp_info[["SNP", "Chromosome", "Position (bp)"]],
    on="SNP",
    how="left"
)

if qtl_snps_pos["Chromosome"].isna().any():
    missing = qtl_snps_pos.loc[qtl_snps_pos["Chromosome"].isna(), "SNP"].head(10).tolist()
    raise ValueError(f"Some QTL_SNPs missing from snp_info. Example missing: {missing}")

# Map each SNP to interval(s) (usually 1 interval; allow multiple if overlaps exist)
snp_to_intervals = []

for _, snp_row in qtl_snps_pos.iterrows():
    snp_id = snp_row["SNP"]
    chr_ = int(snp_row["Chromosome"])
    pos = int(snp_row["Position (bp)"])

    hits = qtl_df_int.loc[
        (qtl_df_int["chrom"] == chr_) &
        (qtl_df_int["start"] <= pos) &
        (qtl_df_int["end"] >= pos),
        ["interval_id", "chrom", "start", "end", "lod_max"]
    ]

    if hits.shape[0] == 0:
        continue

    for _, h in hits.iterrows():
        snp_to_intervals.append({
            "QTL_SNP": snp_id,
            "interval_id": int(h["interval_id"]),
            "qtl_chrom": int(h["chrom"]),
            "qtl_start": int(h["start"]),
            "qtl_end": int(h["end"]),
            "interval_lod_max": float(h["lod_max"]) if "lod_max" in hits.columns else np.nan
        })

map_df = pd.DataFrame(snp_to_intervals)

print("Mapped QTL_SNPs to intervals:", map_df["QTL_SNP"].nunique(), "QTL SNPs")
print("Total SNP->interval mappings:", map_df.shape[0])

# 2) Merge interval_id onto epistasis interactions
epi_iv = epi_clean.merge(map_df[["QTL_SNP", "interval_id"]], on="QTL_SNP", how="left")

if epi_iv["interval_id"].isna().any():
    n_missing = epi_iv["interval_id"].isna().sum()
    print("WARNING: interactions with unmapped QTL_SNP (interval_id NA):", n_missing)

# 3) Interval-level summary
interval_summary = (
    epi_iv.dropna(subset=["interval_id"])
    .groupby("interval_id")
    .agg(
        n_interactions=("P", "count"),
        n_unique_qtl_snps=("QTL_SNP", "nunique"),
        n_unique_mod_snps=("MOD_SNP", "nunique"),
        min_p=("P", "min"),
        max_beta=("BETA_INT", lambda x: x.abs().max())
    )
    .sort_values("n_interactions", ascending=False)
)

interval_summary.head(15)

Mapped QTL_SNPs to intervals: 1518 QTL SNPs
Total SNP->interval mappings: 9347


,n_interactions,n_unique_qtl_snps,n_unique_mod_snps,min_p,max_beta
interval_id,,,,,
38136,739430,806,1686,1.600000e-36,0.019556
38135,739430,806,1686,1.600000e-36,0.019556
38133,739430,806,1686,1.600000e-36,0.019556
38137,739430,806,1686,1.600000e-36,0.019556
38140,739430,806,1686,1.600000e-36,0.019556
38139,739430,806,1686,1.600000e-36,0.019556
38138,739430,806,1686,1.600000e-36,0.019556
38762,559405,440,1686,1.600000e-36,0.019556
38764,559405,440,1686,1.600000e-36,0.019556


## Load Gene Annotation Table

We load the gene annotation file containing:

- gene_name (systematic ORF)
- Standard_name (common gene name)
- chrom
- gene_chr_start
- gene_chr_end

This will allow us to map modifier loci and QTL intervals to genes.

In [19]:
# Code Cell 16: Load gene annotation file
GENE_INFO_PATH = "/blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_epistasis/data/gene_info.csv"

gene_info = pd.read_csv(GENE_INFO_PATH)

# Standardize column types
gene_info["chrom"] = gene_info["chrom"].astype(int)
gene_info["gene_chr_start"] = gene_info["gene_chr_start"].astype(int)
gene_info["gene_chr_end"] = gene_info["gene_chr_end"].astype(int)

print("Gene table shape:", gene_info.shape)
gene_info.head()

Gene table shape: (5190, 5)


,gene_name,chrom,gene_chr_start,gene_chr_end,Standard_name
0,YAL063C,1,24000,27968,FLO9
1,YAL062W,1,31567,32940,GDH3
2,YAL061W,1,33448,34701,BDH2
3,YAL060W,1,35155,36303,BDH1
4,YAL059C-A,1,36496,36918,ECM1


## Map Modifier Loci to Genes

For each collapsed modifier locus (chromosome, start, end),
we identify genes whose genomic coordinates overlap the locus.

Overlap rule:
gene_chr_start ≤ locus_end
AND
gene_chr_end ≥ locus_start

This yields candidate modifier genes driving epistasis.

In [20]:
# Build locus dataframe from earlier loci list
locus_df = pd.DataFrame([
    {"MOD_LOCUS": i, "Chromosome": chr_, "Start": start, "End": end}
    for i, (chr_, start, end) in enumerate(loci)
])

# Merge locus strength information
locus_df = locus_df.merge(
    locus_summary.reset_index(),
    on="MOD_LOCUS",
    how="left"
)

# Map genes overlapping each modifier locus
mod_locus_genes = []

for _, row in locus_df.iterrows():
    chr_ = row["Chromosome"]
    start = row["Start"]
    end = row["End"]

    genes = gene_info.loc[
        (gene_info["chrom"] == chr_) &
        (gene_info["gene_chr_start"] <= end) &
        (gene_info["gene_chr_end"] >= start)
    ]

    for _, g in genes.iterrows():
        mod_locus_genes.append({
            "MOD_LOCUS": row["MOD_LOCUS"],
            "Chromosome": chr_,
            "Locus_Start": start,
            "Locus_End": end,
            "n_interactions": row["n_interactions"],
            "min_p": row["min_p"],
            "max_beta": row["max_beta"],
            "gene_name": g["gene_name"],
            "Standard_name": g["Standard_name"]
        })

mod_locus_genes_df = pd.DataFrame(mod_locus_genes)

print("Total modifier genes identified:", mod_locus_genes_df.shape[0])
mod_locus_genes_df.head(20)

Total modifier genes identified: 518


,MOD_LOCUS,Chromosome,Locus_Start,Locus_End,n_interactions,min_p,max_beta,gene_name,Standard_name
0,0.0,7.0,329109.0,452796.0,121076.0,7.735000e-100,0.029101,YGL095C,VPS45
1,0.0,7.0,329109.0,452796.0,121076.0,7.735000e-100,0.029101,YGL094C,PAN2
2,0.0,7.0,329109.0,452796.0,121076.0,7.735000e-100,0.029101,YGL093W,SPC105
3,0.0,7.0,329109.0,452796.0,121076.0,7.735000e-100,0.029101,YGL092W,NUP145
4,0.0,7.0,329109.0,452796.0,121076.0,7.735000e-100,0.029101,YGL091C,NBP35
5,0.0,7.0,329109.0,452796.0,121076.0,7.735000e-100,0.029101,YGL090W,LIF1
6,0.0,7.0,329109.0,452796.0,121076.0,7.735000e-100,0.029101,YGL089C,MF(ALPHA)2
7,0.0,7.0,329109.0,452796.0,121076.0,7.735000e-100,0.029101,YGL088W,YGL088W
8,0.0,7.0,329109.0,452796.0,121076.0,7.735000e-100,0.029101,YGL086W,MAD1
9,0.0,7.0,329109.0,452796.0,121076.0,7.735000e-100,0.029101,YGL085W,LCL3


## Inspect Top Modifier Locus

This locus had:
- ~91,000 interactions
- Strongest interaction P-value
- Largest |β|

We extract genes overlapping this locus specifically.

In [21]:
# Code Cell — Inspect dominant modifier locus

top_locus_id = locus_summary.index[0]

top_locus = locus_df.loc[locus_df["MOD_LOCUS"] == top_locus_id].iloc[0]

print("Top Locus ID:", top_locus_id)
print(top_locus)

top_genes = mod_locus_genes_df[
    mod_locus_genes_df["MOD_LOCUS"] == top_locus_id
]

print("Number of genes in this locus:", top_genes.shape[0])
top_genes.head(20)

Top Locus ID: 9
MOD_LOCUS          9.000000e+00
Chromosome         1.400000e+01
Start              3.563550e+05
End                5.740370e+05
n_interactions     5.690060e+05
min_p             8.028000e-130
max_beta           3.283550e-02
Name: 9, dtype: float64
Number of genes in this locus: 96


,MOD_LOCUS,Chromosome,Locus_Start,Locus_End,n_interactions,min_p,max_beta,gene_name,Standard_name
284,9.0,14.0,356355.0,574037.0,569006.0,8.028000e-130,0.032835,YNL143C,YNL143C
285,9.0,14.0,356355.0,574037.0,569006.0,8.028000e-130,0.032835,YNL142W,MEP2
286,9.0,14.0,356355.0,574037.0,569006.0,8.028000e-130,0.032835,YNL139C,THO2
287,9.0,14.0,356355.0,574037.0,569006.0,8.028000e-130,0.032835,YNL138W-A,YSF3
288,9.0,14.0,356355.0,574037.0,569006.0,8.028000e-130,0.032835,YNL138W,SRV2
289,9.0,14.0,356355.0,574037.0,569006.0,8.028000e-130,0.032835,YNL137C,NAM9
290,9.0,14.0,356355.0,574037.0,569006.0,8.028000e-130,0.032835,YNL136W,EAF7
291,9.0,14.0,356355.0,574037.0,569006.0,8.028000e-130,0.032835,YNL135C,FPR1
292,9.0,14.0,356355.0,574037.0,569006.0,8.028000e-130,0.032835,YNL134C,YNL134C
293,9.0,14.0,356355.0,574037.0,569006.0,8.028000e-130,0.032835,YNL133C,FYV6


## Refine Modifier Locus Clustering (5 kb window)

The dominant modifier locus spans ~350 kb,
which is too large to represent a single functional unit.

We reduce clustering window from 10 kb → 5 kb
to split potential independent LD blocks.

In [22]:
# Re-cluster modifier SNPs with smaller window
WINDOW = 5_000

mod_annot_sorted = mod_annot.sort_values(
    ["Chromosome", "Position (bp)"]
).reset_index(drop=True)

loci_refined = []
current_chr = None
current_start = None
current_end = None

for _, row in mod_annot_sorted.iterrows():
    chr_ = row["Chromosome"]
    pos = row["Position (bp)"]

    if current_chr is None:
        current_chr = chr_
        current_start = pos
        current_end = pos
    elif chr_ == current_chr and pos - current_end <= WINDOW:
        current_end = pos
    else:
        loci_refined.append((current_chr, current_start, current_end))
        current_chr = chr_
        current_start = pos
        current_end = pos

if current_chr is not None:
    loci_refined.append((current_chr, current_start, current_end))

print("Independent modifier loci (5kb clustering):", len(loci_refined))
loci_refined[:10]

Independent modifier loci (5kb clustering): 27


[(7, 329109, 452796),
 (7, 516638, 535692),
 (7, 541712, 561756),
 (7, 568033, 568674),
 (7, 574724, 678180),
 (7, 754648, 754648),
 (7, 785199, 800053),
 (7, 832207, 903446),
 (8, 57484, 85218),
 (8, 92147, 101018)]

## Recompute Locus-Level Strength After 5 kb Clustering

We now:
1. Reassign modifier SNPs to refined loci
2. Aggregate interaction statistics per refined locus
3. Identify which refined loci dominate epistasis

In [23]:
# Build refined locus dataframe
locus_df_refined = pd.DataFrame([
    {"MOD_LOCUS": i, "Chromosome": chr_, "Start": start, "End": end}
    for i, (chr_, start, end) in enumerate(loci_refined)
])

# Create new SNP -> locus map
locus_map_refined = {}

for i, (chr_, start, end) in enumerate(loci_refined):
    snps_in_locus = mod_annot[
        (mod_annot["Chromosome"] == chr_) &
        (mod_annot["Position (bp)"] >= start) &
        (mod_annot["Position (bp)"] <= end)
    ]["MOD_SNP"].tolist()
    
    for s in snps_in_locus:
        locus_map_refined[s] = i

epi_clean["MOD_LOCUS_REFINED"] = epi_clean["MOD_SNP"].map(locus_map_refined)

# Aggregate refined locus strength
locus_summary_refined = (
    epi_clean
    .groupby("MOD_LOCUS_REFINED")
    .agg(
        n_interactions=("P", "count"),
        min_p=("P", "min"),
        max_beta=("BETA_INT", lambda x: x.abs().max())
    )
    .sort_values("n_interactions", ascending=False)
)

print("Number of refined loci:", locus_summary_refined.shape[0])
locus_summary_refined.head(10)

Number of refined loci: 27


,n_interactions,min_p,max_beta
MOD_LOCUS_REFINED,,,
15,241217,8.028000e-130,0.032835
17,196293,1.808000e-34,0.014564
0,121076,7.735000e-100,0.029101
23,116853,3.215000e-22,0.013816
16,114501,4.988000e-107,0.029135
2,70837,7.545000e-91,0.027874
9,61110,9.960000e-30,0.016619
8,55602,1.874000e-23,0.014691
1,54455,1.179000e-106,0.029956


## Map Top Refined Modifier Loci to Genes

We now focus on the top refined modifier loci (by interaction count).

This allows us to identify candidate modifier genes driving epistasis.

In [24]:
# Take top 5 refined loci
top_refined_ids = locus_summary_refined.head(5).index.tolist()

top_locus_df = locus_df_refined[
    locus_df_refined["MOD_LOCUS"].isin(top_refined_ids)
]

refined_genes = []

for _, row in top_locus_df.iterrows():
    chr_ = row["Chromosome"]
    start = row["Start"]
    end = row["End"]
    locus_id = row["MOD_LOCUS"]

    genes = gene_info.loc[
        (gene_info["chrom"] == chr_) &
        (gene_info["gene_chr_start"] <= end) &
        (gene_info["gene_chr_end"] >= start)
    ]

    for _, g in genes.iterrows():
        refined_genes.append({
            "MOD_LOCUS": locus_id,
            "Chromosome": chr_,
            "Start": start,
            "End": end,
            "gene_name": g["gene_name"],
            "Standard_name": g["Standard_name"]
        })

refined_genes_df = pd.DataFrame(refined_genes)

print("Genes in top refined loci:", refined_genes_df.shape[0])
refined_genes_df.sort_values(["MOD_LOCUS"]).head(30)

Genes in top refined loci: 195


,MOD_LOCUS,Chromosome,Start,End,gene_name,Standard_name
0,0,7,329109,452796,YGL095C,VPS45
1,0,7,329109,452796,YGL094C,PAN2
2,0,7,329109,452796,YGL093W,SPC105
3,0,7,329109,452796,YGL092W,NUP145
4,0,7,329109,452796,YGL091C,NBP35
5,0,7,329109,452796,YGL090W,LIF1
6,0,7,329109,452796,YGL089C,MF(ALPHA)2
7,0,7,329109,452796,YGL088W,YGL088W
8,0,7,329109,452796,YGL086W,MAD1
9,0,7,329109,452796,YGL085W,LCL3


## Identify Strongest Modifier SNPs Within Top Locus

We now focus on modifier SNPs within the top refined locus
and identify the most epistatically active SNPs.

This helps narrow candidate genes inside the large locus.

In [25]:
# Get SNPs in top locus
top_locus_id = 2

top_mod_snps = epi_clean.loc[
    epi_clean["MOD_LOCUS_REFINED"] == top_locus_id,
    "MOD_SNP"
].unique()

top_mod_df = pd.DataFrame({"MOD_SNP": top_mod_snps})

# Merge genomic position
top_mod_df = top_mod_df.merge(
    snp_info[["SNP", "Chromosome", "Position (bp)"]],
    left_on="MOD_SNP",
    right_on="SNP",
    how="left"
)

# Count interactions per SNP
snp_strength = (
    epi_clean[epi_clean["MOD_LOCUS_REFINED"] == top_locus_id]
    .groupby("MOD_SNP")
    .agg(
        n_interactions=("P", "count"),
        min_p=("P", "min"),
        max_beta=("BETA_INT", lambda x: x.abs().max())
    )
    .sort_values("n_interactions", ascending=False)
)

snp_strength.head(10)

,n_interactions,min_p,max_beta
MOD_SNP,,,
snp14793,757,2.673000e-83,0.026760
snp14794,757,2.936000e-83,0.026760
snp14800,752,1.559000e-81,0.026503
snp14797,752,1.492000e-81,0.026507
snp14796,752,1.492000e-81,0.026507
snp14798,752,1.559000e-81,0.026503
snp14799,752,1.559000e-81,0.026503
snp14768,739,2.097000e-86,0.027210
snp14792,739,1.384000e-84,0.026951


## Localize Top Modifier SNP Block

We map the strongest modifier SNPs
to genomic position and overlapping genes
to identify the likely causal modifier gene.

In [26]:
# Take top 15 SNPs
top_snp_ids = snp_strength.head(15).index.tolist()

top_snp_df = pd.DataFrame({"MOD_SNP": top_snp_ids})

# Merge genomic position
top_snp_df = top_snp_df.merge(
    snp_info[["SNP", "Chromosome", "Position (bp)"]],
    left_on="MOD_SNP",
    right_on="SNP",
    how="left"
)

top_snp_df = top_snp_df.drop(columns=["SNP"])

top_snp_df

,MOD_SNP,Chromosome,Position (bp)
0,snp14793,7,557243
1,snp14794,7,557597
2,snp14800,7,558969
3,snp14797,7,558809
4,snp14796,7,558800
5,snp14798,7,558857
6,snp14799,7,558946
7,snp14768,7,550438
8,snp14792,7,556096
9,snp14767,7,549847


## Identify Candidate Gene(s) in Strong Modifier Block

We now identify genes overlapping the narrow SNP block driving the strongest epistatic signal.

In [27]:
# Define narrow block boundaries
block_chr = 7
block_start = 329109
block_end = 452796

block_genes = gene_info.loc[
    (gene_info["chrom"] == block_chr) &
    (gene_info["gene_chr_start"] <= block_end) &
    (gene_info["gene_chr_end"] >= block_start)
]

block_genes.sort_values("gene_chr_start")

,gene_name,chrom,gene_chr_start,gene_chr_end,Standard_name
1705,YGL095C,7,328874,330607,VPS45
1706,YGL094C,7,331118,334465,PAN2
1707,YGL093W,7,334886,337639,SPC105
1708,YGL092W,7,337906,341859,NUP145
1709,YGL091C,7,342056,343042,NBP35
...,...,...,...,...,...
1764,YGL027C,7,443642,446143,CWH41
1765,YGL026C,7,446412,448535,TRP5
1766,YGL025C,7,448764,449957,PGD1
1767,YGL023C,7,450197,452104,PIB2


## Save processed outputs for Notebook 05

This section exports standardized tables from the post-processing workflow so that
the figure notebook can load them directly without repeating the full analysis.

Saved outputs include:
- cleaned significant epistasis interactions
- modifier SNP summaries
- modifier locus summaries
- modifier locus gene annotations
- QTL interval mappings
- interval-level interaction summaries

In [33]:
# Save processed outputs for 05 notebook
from pathlib import Path

OUT_DIR = Path("output/processed_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

PREFIX = f"{drug_code}_{RUN_LABEL}"

print("Output directory:", OUT_DIR)
print("Prefix:", PREFIX)

Output directory: output/processed_outputs
Prefix: flu_all_x_qtl


In [34]:
required_objects = [
    "epi_clean",
    "mod_summary",
    "mod_annot",
    "locus_df",
    "locus_summary",
    "mod_locus_genes_df",
]

missing_objects = [x for x in required_objects if x not in globals()]
if missing_objects:
    raise ValueError(f"Missing required objects before export: {missing_objects}")

print("All required core objects are present.")

All required core objects are present.


In [35]:
# Make copies so display objects in notebook are not altered unintentionally
epi_clean_export = epi_clean.copy()
mod_summary_export = mod_summary.copy()
mod_annot_export = mod_annot.copy()
locus_df_export = locus_df.copy()
locus_summary_export = locus_summary.copy()
mod_locus_genes_export = mod_locus_genes_df.copy()

# Clean common integer-like columns if present
for df in [locus_df_export, locus_summary_export, mod_locus_genes_export]:
    for col in ["MOD_LOCUS", "Chromosome", "Start", "End", "Locus_Start", "Locus_End"]:
        if col in df.columns:
            try:
                df[col] = df[col].astype("Int64")
            except Exception:
                pass

for df in [epi_clean_export, mod_annot_export]:
    for col in ["CHR1", "CHR2", "Chromosome", "Position (bp)"]:
        if col in df.columns:
            try:
                df[col] = df[col].astype("Int64")
            except Exception:
                pass

In [36]:
# Core epistasis result table used for many downstream figures
epi_clean_export.to_csv(
    OUT_DIR / f"{PREFIX}_epi_clean.tsv",
    sep="\t",
    index=False
)

# Modifier SNP summaries
mod_summary_export.to_csv(
    OUT_DIR / f"{PREFIX}_modifier_snp_summary.tsv",
    sep="\t",
    index=False
)

mod_annot_export.to_csv(
    OUT_DIR / f"{PREFIX}_modifier_snp_annotated.tsv",
    sep="\t",
    index=False
)

# Modifier locus tables
locus_df_export.to_csv(
    OUT_DIR / f"{PREFIX}_modifier_loci.tsv",
    sep="\t",
    index=False
)

# IMPORTANT FIX: preserve MOD_LOCUS as a real column
locus_summary_export.reset_index().to_csv(
    OUT_DIR / f"{PREFIX}_modifier_locus_summary.tsv",
    sep="\t",
    index=False
)

# Modifier locus -> genes
mod_locus_genes_export.to_csv(
    OUT_DIR / f"{PREFIX}_modifier_locus_genes.tsv",
    sep="\t",
    index=False
)

# IMPORTANT FIX: preserve interval_id as a real column
if "interval_summary" in globals():
    interval_summary.reset_index().to_csv(
        OUT_DIR / f"{PREFIX}_interval_summary.tsv",
        sep="\t",
        index=False
    )

print("Core tables saved.")

Core tables saved.


In [37]:
optional_objects = {
    "map_df": f"{PREFIX}_qtl_snp_to_interval.tsv",
    "epi_iv": f"{PREFIX}_epi_with_interval.tsv",
}

saved_optional = []
missing_optional = []

for obj_name, fname in optional_objects.items():
    if obj_name in globals():
        obj = globals()[obj_name].copy()
        obj.to_csv(OUT_DIR / fname, sep="\t", index=False)
        saved_optional.append(fname)
    else:
        missing_optional.append(obj_name)

print("Optional saved:", saved_optional if saved_optional else "None")
print("Optional missing:", missing_optional if missing_optional else "None")

Optional saved: ['flu_all_x_qtl_qtl_snp_to_interval.tsv', 'flu_all_x_qtl_epi_with_interval.tsv']
Optional missing: None


In [38]:
EPI_SUMMARY = DATA_DIR / f"epi_{drug_code}_{RUN_LABEL}.epi.qt.summary"

meta_df = pd.DataFrame({
    "key": [
        "drug",
        "run_label",
        "epi_qt_file",
        "epi_summary_file",
        "set_file",
        "n_epi_rows",
        "n_epi_clean_rows",
        "n_unique_qtl_snps",
        "n_unique_mod_snps",
        "bonf_threshold",
    ],
    "value": [
        DRUG if "DRUG" in globals() else "NA",
        RUN_LABEL if "RUN_LABEL" in globals() else "NA",
        str(EPI_QT) if "EPI_QT" in globals() else "NA",
        str(EPI_SUMMARY) if "EPI_SUMMARY" in globals() else "NA",
        str(SET_FILE) if "SET_FILE" in globals() else "NA",
        epi.shape[0] if "epi" in globals() else np.nan,
        epi_clean.shape[0] if "epi_clean" in globals() else np.nan,
        epi_clean["QTL_SNP"].nunique() if "epi_clean" in globals() else np.nan,
        epi_clean["MOD_SNP"].nunique() if "epi_clean" in globals() else np.nan,
        f"{bonf_threshold:.6e}" if "bonf_threshold" in globals() else "NA",
    ]
})

meta_df.to_csv(
    OUT_DIR / f"{PREFIX}_metadata.tsv",
    sep="\t",
    index=False
)

meta_df

,key,value
0,drug,FLU
1,run_label,all_x_qtl
2,epi_qt_file,/blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_e...
3,epi_summary_file,/blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_e...
4,set_file,/blue/juannanzhou/arindam.sarkar/BB-QTL_HT/5_e...
5,n_epi_rows,4665905
6,n_epi_clean_rows,1296604
7,n_unique_qtl_snps,1518
8,n_unique_mod_snps,4045
9,bonf_threshold,4.266969e-10


In [40]:
saved_files = sorted(OUT_DIR.glob(f"{PREFIX}_*"))
print("Saved files:")
for f in saved_files:
    print(f.name)

Saved files:
flu_all_x_qtl_epi_clean.tsv
flu_all_x_qtl_epi_with_interval.tsv
flu_all_x_qtl_interval_summary.tsv
flu_all_x_qtl_metadata.tsv
flu_all_x_qtl_modifier_loci.tsv
flu_all_x_qtl_modifier_locus_genes.tsv
flu_all_x_qtl_modifier_locus_summary.tsv
flu_all_x_qtl_modifier_snp_annotated.tsv
flu_all_x_qtl_modifier_snp_summary.tsv
flu_all_x_qtl_qtl_snp_to_interval.tsv
